# Build 1: LoRA fine-tuning, end to end

**Goal:** understand every step of fine-tuning Llama-2-7B on SQuAD with LoRA,
on a *tiny* model so it runs in seconds.

**What is LoRA?** We freeze the base weight $W$ and learn a small low-rank update:

$$W' = W + \frac{\alpha}{r}\, B A,\quad A\in\mathbb{R}^{d\times r},\; B\in\mathbb{R}^{r\times d},\; r\ll d$$

Only $A,B$ train. With $r=80,\ \alpha=160$ this matches HiMoLE's trainable-param budget.

> Cells that touch a `# TODO` will raise `NotImplementedError` until you fill it.
> That is expected — the notebook shows you *where* each learning point lives.

In [ ]:
from himole.config import BaselineConfig

cfg = BaselineConfig()
cfg.use_tiny = True          # sshleifer/tiny-gpt2, runs on CPU
cfg.max_train_samples = 8    # keep it tiny
cfg.max_steps = 2
cfg

## Step 1 — Data & the loss mask

We turn a SQuAD example into `input_ids` (prompt + answer) and `labels`.
The trick: set `labels = -100` on every **prompt** token so the loss only
scores the **answer**. (PyTorch cross-entropy ignores `-100`.)

**TODO sites:** `build_prompt` and `format_example` in `himole/data/squad.py`.

In [ ]:
from transformers import AutoTokenizer
from himole.data.squad import build_prompt, format_example

tok = AutoTokenizer.from_pretrained(cfg.tiny_model)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

ex = {'context': 'Paris is the capital of France.',
      'question': 'What is the capital of France?',
      'answers': {'text': ['Paris']}}

out = format_example(ex, tok, cutoff_len=64)   # raises until you fill the TODO
for tid, lab in zip(out['input_ids'], out['labels']):
    print(f"{tid:>6}  label={lab}")           # label=-100 means 'masked / no loss'

## Step 2 — Attach LoRA

Freeze the base model, add LoRA adapters on the FFN projections.
`print_trainable_parameters()` shows how tiny the trainable slice is.

**TODO site:** `attach_lora` in `himole/model/baseline_lora.py`.

In [ ]:
from himole.model.baseline_lora import load_base_model, attach_lora

model = attach_lora(load_base_model(cfg), cfg)   # raises until you fill the TODO

## Step 3 — One training step

The whole loop is just: `outputs = model(**batch)` → `loss = outputs.loss`
→ `loss.backward()` → `optimizer.step()` → `optimizer.zero_grad()`.

**TODO site:** the core step in `himole/train/loop.py`.

In [ ]:
from himole.data.squad import load_squad
from himole.train.loop import train

train_ds, _ = load_squad(tok, cfg)
id_pairs = [{'prompt': build_prompt(ex['context'], ex['question']),
             'gold': ex['answers']['text'][0]}]

train(model, tok, train_ds, id_pairs, cfg)       # raises until TODOs are filled

## Step 4 — Evaluate EM / ROUGE-2

Generate answers and score them. **TODO sites:** `himole/eval/metrics.py`.

In [ ]:
from himole.eval.generate import evaluate

evaluate(model, tok, id_pairs)                   # raises until metrics TODOs are filled

## Your homework — fill the TODOs in this order

Each one has a test that turns green when you get it right:

1. `himole/data/squad.py` → `build_prompt`, `format_example`  —  `pytest tests/test_data_masking.py`
2. `himole/eval/metrics.py` → normalize / EM / ROUGE-2 / score_batch  —  `pytest tests/test_metrics.py`
3. `himole/model/baseline_lora.py` → `attach_lora` (the `LoraConfig`)
4. `himole/train/loop.py` → the forward/backward/step
5. `himole/data/newsqa.py` → the OOD loader

Then run the smoke test end-to-end:
`python scripts/train_baseline.py --tiny --steps 5 --samples 8`